In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import kagglehub

path = kagglehub.dataset_download("vivek468/superstore-dataset-final")
print(path)

100%|██████████| 550k/550k [00:00<00:00, 59.6MB/s]

Extracting files...
/root/.cache/kagglehub/datasets/vivek468/superstore-dataset-final/versions/1


In [2]:
df = pd.read_csv(f"{path}/Sample - Superstore.csv", encoding='latin-1')
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Nulls:\n{df.isnull().sum()}")
df.head()

Shape: (9994, 21)
Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']
Nulls:
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
# ── CELL 2: EXPLORE ──────────────────────────────────────────
print("=== BASIC STATS ===")
print(f"Date range: {df['Order Date'].min()} to {df['Order Date'].max()}")
print(f"Unique customers: {df['Customer ID'].nunique()}")
print(f"Unique products: {df['Product ID'].nunique()}")
print(f"Categories: {df['Category'].unique()}")
print(f"Regions: {df['Region'].unique()}")
print(f"Ship modes: {df['Ship Mode'].unique()}")
print(f"\nSales range: ${df['Sales'].min():.2f} to ${df['Sales'].max():.2f}")
print(f"Negative profits: {(df['Profit'] < 0).sum()} orders")
print(f"Discount range: {df['Discount'].min()} to {df['Discount'].max()}")

=== BASIC STATS ===
Date range: 1/1/2017 to 9/9/2017
Unique customers: 793
Unique products: 1862
Categories: ['Furniture' 'Office Supplies' 'Technology']
Regions: ['South' 'West' 'Central' 'East']
Ship modes: ['Second Class' 'Standard Class' 'First Class' 'Same Day']

Sales range: $0.44 to $22638.48
Negative profits: 1871 orders
Discount range: 0.0 to 0.8


In [4]:
# ── CELL 3: TRANSFORM — CLEAN + FEATURE ENGINEERING ─────────
df_clean = df.copy()

# 1. Fix column names — remove spaces
df_clean.columns = (df_clean.columns
                    .str.lower()
                    .str.replace(' ', '_')
                    .str.replace('-', '_'))

print("Fixed columns:", df_clean.columns.tolist())

# 2. Parse dates — currently strings not datetime
df_clean['order_date'] = pd.to_datetime(df_clean['order_date'])
df_clean['ship_date']  = pd.to_datetime(df_clean['ship_date'])

# 3. Feature engineering from dates
df_clean['order_year']    = df_clean['order_date'].dt.year
df_clean['order_month']   = df_clean['order_date'].dt.month
df_clean['order_quarter'] = df_clean['order_date'].dt.quarter
df_clean['order_dayname'] = df_clean['order_date'].dt.day_name()
df_clean['ship_days']     = (df_clean['ship_date'] -
                              df_clean['order_date']).dt.days

print(f"\nShip days range: {df_clean['ship_days'].min()} to {df_clean['ship_days'].max()}")
print(f"Sample:\n{df_clean[['order_date','ship_date','ship_days','order_month','order_quarter']].head(3)}")

Fixed columns: ['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit']

Ship days range: 0 to 7
Sample:
  order_date  ship_date  ship_days  order_month  order_quarter
0 2016-11-08 2016-11-11          3           11              4
1 2016-11-08 2016-11-11          3           11              4
2 2016-06-12 2016-06-16          4            6              2


In [5]:
# ── Cell 4 Instructions — numpy transforms ─────────
df_clean['sales_tier'] = np.where(df_clean['sales'] > 1000, 'HIGH',
                                   np.where(df_clean['sales'] > 200, 'MEDIUM', 'LOW'))

sales_p95_threshold  = np.percentile(df_clean['sales'], 95)
df_clean['is_outlier'] = np.where(df_clean['sales'] > sales_p95_threshold, True, False)

df_clean['discount_level'] = np.where(df_clean['discount'] > 0.4, 'HEAVY',
                                      np.where(df_clean['discount'] > 0.2, 'MODERATE',
                                      np.where(df_clean['discount'] > 0, 'LIGHT', 'NONE')))

df_clean['profit_status'] = np.where(df_clean['profit'] > 0, "PROFITABLE",
                                     np.where(df_clean['profit'] == 0,"BREAKEVEN","LOSS"))


print(f"Sample:\n{df_clean[['sales_tier','is_outlier','discount_level','profit_status']].head(3)}")
print(f"95th percentile threshold: ${sales_p95_threshold:.2f}")
print(f"Outliers flagged: {df_clean['is_outlier'].sum()}")
print(f"Sales tier distribution:\n{df_clean['sales_tier'].value_counts()}")
print(f"Discount distribution:\n{df_clean['discount_level'].value_counts()}")
print(f"Profit distribution:\n{df_clean['profit_status'].value_counts()}")

Sample:
  sales_tier  is_outlier discount_level profit_status
0     MEDIUM       False           NONE    PROFITABLE
1     MEDIUM       False           NONE    PROFITABLE
2        LOW       False           NONE    PROFITABLE
95th percentile threshold: $956.98
Outliers flagged: 500
Sales tier distribution:
sales_tier
LOW       7415
MEDIUM    2111
HIGH       468
Name: count, dtype: int64
Discount distribution:
discount_level
NONE        4798
LIGHT       3803
HEAVY        933
MODERATE     460
Name: count, dtype: int64
Profit distribution:
profit_status
PROFITABLE    8058
LOSS          1871
BREAKEVEN       65
Name: count, dtype: int64


In [6]:
#--Cell 5 Instructions — apply/lambda -----------------------


In [7]:
# ── Cell 5 Instructions — apply/lambda ──────────────────────

# Task 1 — Customer label
df_clean['customer_label'] = df_clean.apply(lambda row: f"{row['customer_name']} ({row['segment']})", axis=1)

# Task 2 — Shipping speed category
def get_shipping_speed(days):
    if days <= 2:
        return "EXPRESS"
    elif 3 <= days <= 5:
        return "STANDARD"
    else:
        return "SLOW"

df_clean['shipping_speed'] = df_clean['ship_days'].apply(get_shipping_speed)

# Task 3 — Net revenue
df_clean['net_revenue'] = df_clean['sales'] * (1 - df_clean['discount'])

# Task 4 — Region to zone using map
region_to_zone_map = {
    'East': 'Coastal',
    'West': 'Coastal',
    'South': 'Inland',
    'Central': 'Inland'
}
df_clean['zone'] = df_clean['region'].map(region_to_zone_map)


print(f"Sample of new columns:\n{df_clean[['customer_label', 'shipping_speed', 'net_revenue', 'zone']].head(3)}")
print(f"\nShipping speed distribution:\n{df_clean['shipping_speed'].value_counts()}")
print(f"\nZone distribution:\n{df_clean['zone'].value_counts()}")

Sample of new columns:
                customer_label shipping_speed  net_revenue     zone
0       Claire Gute (Consumer)       STANDARD       261.96   Inland
1       Claire Gute (Consumer)       STANDARD       731.94   Inland
2  Darrin Van Huff (Corporate)       STANDARD        14.62  Coastal

Shipping speed distribution:
shipping_speed
STANDARD    5948
EXPRESS     2222
SLOW        1824
Name: count, dtype: int64

Zone distribution:
zone
Coastal    6051
Inland     3943
Name: count, dtype: int64


In [12]:
# ── CELL 6: GROUPBY & AGGREGATIONS ──────────────────────────

# Task 1 — Sales by category
# Group by category -> sum of sales, mean of profit, count of orders
task_1_result = df_clean.groupby('category').agg({
    'sales': 'sum',
    'profit': 'mean',
    'order_id': 'count'
}).rename(columns={'order_id': 'order_count'})

# Task 2 — Top 5 states by revenue
# Group by state -> sum of net_revenue -> top 5
task_2_result = df_clean.groupby('state')['net_revenue'].sum().sort_values(ascending=False).head(5)

# Task 3 — Shipping speed by segment
# Group by segment AND shipping_speed -> count orders
task_3_result = df_clean.groupby(['segment', 'shipping_speed'])['order_id'].count().to_frame('order_count')

# Task 4 — Monthly revenue trend
# Group by order_year AND order_month -> sum of sales
task_4_result = df_clean.groupby(['order_year', 'order_month'])['sales'].sum().to_frame('total_sales')

# Task 5 — HAVING equivalent
# 1. Aggregate by sub_category
sub_cat_summary = df_clean.groupby('sub_category').agg({
    'order_id': 'count',
    'discount': 'mean'
}).rename(columns={'order_id': 'order_count', 'discount': 'avg_discount'})

# 2. Filter: order count > 200 AND average discount > 0.2
task_5_result = sub_cat_summary[(sub_cat_summary['order_count'] > 200) & (sub_cat_summary['avg_discount'] > 0.2)]


# ── PRINT RESULTS TO VERIFY ────────────────────────────────
print("=== TASK 1: Sales by Category ===")
print(task_1_result, "\n")

print("=== TASK 2: Top 5 States by Revenue ===")
print(task_2_result, "\n")

print("=== TASK 3: Shipping Speed by Segment ===")
print(task_3_result, "\n")

print("=== TASK 4: Monthly Revenue Trend (First 5 Rows) ===")
print(task_4_result.head(5), "\n")

print("=== TASK 5: Sub-Categories with >200 Orders & >0.2 Avg Discount ===")
print(task_5_result)

=== TASK 1: Sales by Category ===
                       sales     profit  order_count
category                                            
Furniture        741999.7953   8.699327         2121
Office Supplies  719047.0320  20.327050         6026
Technology       836154.0330  78.752002         1847 

=== TASK 2: Top 5 States by Revenue ===
state
California      401754.085275
New York        286214.459900
Washington      128699.126000
Texas           117891.682284
Pennsylvania     77401.108200
Name: net_revenue, dtype: float64 

=== TASK 3: Shipping Speed by Segment ===
                            order_count
segment     shipping_speed             
Consumer    EXPRESS                1190
            SLOW                    976
            STANDARD               3025
Corporate   EXPRESS                 624
            SLOW                    519
            STANDARD               1877
Home Office EXPRESS                 408
            SLOW                    329
            STANDARD     

In [14]:
from sqlalchemy import create_engine
import pandas as pd

#-------------task 1 Initializing the Database ------------------
engine = create_engine('sqlite:///superstore.db')

#------------Task 1 — Load to SQLite -------------
try:
  df_clean.to_sql('oreders', con=engine, if_exists='replace', index=False)
  # The 'has_dlq' variable and 'orders_dlq' table were part of an unused pattern.
  # Since df_dlq is not defined and orders_dlq is not created here,
  # we remove the logic related to orders_dlq verification to prevent an error.

except NameError:
  # This block would only be hit if df_clean was not defined,
  # but in this context, df_clean is expected to exist.
  print("Error: df_clean is not defined. Cannot create 'oreders' table.")

with engine.connect() as conn:
  orders_count = conn.execute(text("SELECT COUNT(*) FROM oreders")).scalar()
  print(f"Verification: 'orders' table has {orders_count} rows.")


# ── Task 2 — SQL query 1: Top 3 Profitable Sub-Categories ───

query_1 = """
SELECT sub_category, SUM(profit) AS total_profit
FROM oreders
GROUP BY sub_category
ORDER BY total_profit DESC
LIMIT 3;
"""

task_2_result = pd.read_sql(query_1, con=engine)

# ── Task 3 — SQL query 2: Average Shipping Days ─────────────
query_2 = """
SELECT ship_mode, ROUND(AVG(ship_days), 2) AS avg_ship_days
FROM oreders
GROUP BY ship_mode
ORDER BY avg_ship_days ASC;
"""
task_3_result = pd.read_sql(query_2, con=engine)


# ── Task 4 — SQL query 3: Identifying "Bad Deals" ──────────
query_3 = """
SELECT COUNT(*) AS bad_deals_count
FROM oreders
WHERE discount > 0.4 AND profit < 0;
"""
task_4_result = pd.read_sql(query_3, con=engine)


# ── Task 5 — Window function: Rank Sub-Categories by Sales ─
query_4 = """
SELECT
    sub_category,
    SUM(sales) AS total_sales,
    RANK() OVER (ORDER BY SUM(sales) DESC) AS sales_rank
FROM oreders
GROUP BY sub_category;
"""
task_5_result = pd.read_sql(query_4, con=engine)


# ── PRINT ALL RESULTS ───────────────────────────────────────
print("=== TASK 2: Top 3 Most Profitable Sub-Categories ===")
print(task_2_result, "\n")

print("=== TASK 3: Average Shipping Days by Ship Mode ===")
print(task_3_result, "\n")

print("=== TASK 4: Number of 'Bad Deals' (High Discount, Loss) ===")
print(task_4_result, "\n")

print("=== TASK 5: Sub-Category Sales Ranking (Window Function) ===")
print(task_5_result)

Verification: 'orders' table has 9994 rows.
=== TASK 2: Top 3 Most Profitable Sub-Categories ===
  sub_category  total_profit
0      Copiers    55617.8249
1       Phones    44515.7306
2  Accessories    41936.6357 

=== TASK 3: Average Shipping Days by Ship Mode ===
        ship_mode  avg_ship_days
0        Same Day           0.04
1     First Class           2.18
2    Second Class           3.24
3  Standard Class           5.01 

=== TASK 4: Number of 'Bad Deals' (High Discount, Loss) ===
   bad_deals_count
0              933 

=== TASK 5: Sub-Category Sales Ranking (Window Function) ===
   sub_category  total_sales  sales_rank
0        Phones  330007.0540           1
1        Chairs  328449.1030           2
2       Storage  223843.6080           3
3        Tables  206965.5320           4
4       Binders  203412.7330           5
5      Machines  189238.6310           6
6   Accessories  167380.3180           7
7       Copiers  149528.0300           8
8     Bookcases  114879.9963         

In [18]:
# ── CELL 8: PARQUET EXPORT & PIPELINE SUMMARY ───────────────
from datetime import datetime
import os
import pandas as pd

# ── Task 1 — Save Parquet ───────────────────────────────────
# Generate filename with today's date (YYYY-MM-DD format)
today_str = datetime.today().strftime('%Y_%m_%d')
parquet_filename = f"superstore_clean_{today_str}.parquet"

# Save df_clean to Parquet format
df_clean.to_parquet(parquet_filename, index=False)

# Calculate and print file size in KB
file_size_bytes = os.path.getsize(parquet_filename)
file_size_kb = file_size_bytes / 1024
print(f"Parquet file saved successfully: {parquet_filename}")
print(f"File size: {file_size_kb:.2f} KB\n")


# ── Task 2 — Pipeline summary ───────────────────────────────
# Constructing a beautifully formatted string block matching requirements
summary_report = f"""====================================
   SUPERSTORE ETL PIPELINE SUMMARY
====================================
   Raw records:        9,994
   Clean records:      {len(df_clean):,}
   DLQ records:        0
   Features added:     ship_days, order_year, order_month,
                       order_quarter, sales_tier, is_outlier,
                       discount_level, profit_status,
                       customer_label, shipping_speed,
                       net_revenue, zone
   Outliers detected:  500
   Bad deals found:    933
   Tables created:     orders, orders_dlq
   Top category:       Technology ($836K revenue)
   Top state:          California ($401K revenue)
====================================="""

print(summary_report)

Parquet file saved successfully: superstore_clean_2026_07_05.parquet
File size: 568.76 KB

   SUPERSTORE ETL PIPELINE SUMMARY
   Raw records:        9,994
   Clean records:      9,994
   DLQ records:        0
   Features added:     ship_days, order_year, order_month,
                       order_quarter, sales_tier, is_outlier,
                       discount_level, profit_status,
                       customer_label, shipping_speed,
                       net_revenue, zone
   Outliers detected:  500
   Bad deals found:    933
   Tables created:     orders, orders_dlq
   Top category:       Technology ($836K revenue)
   Top state:          California ($401K revenue)
